---
## 💾 How to Store the Model on Kaggle Permanently

Kaggle's `/kaggle/working/` directory is **temporary** — it disappears after the session ends. Use one of these methods to keep your model:

### ✅ Method 1 — Kaggle Models (Recommended)
1. Go to **kaggle.com → Models → New Model**
2. Upload the files from `/kaggle/working/eps-lora-adapters/`
3. In future notebooks, access via: `/kaggle/input/<your-model-slug>/`

### ✅ Method 2 — Kaggle Datasets
```python
# Run in terminal / notebook
!kaggle datasets create -p /kaggle/working/eps-lora-adapters --name eps-forecast-lora
# Later versions:
!kaggle datasets version -p /kaggle/working/eps-lora-adapters -m "v2 - 3 epochs"
```

### ✅ Method 3 — Hugging Face Hub (Cell 10 above)
Free, permanent, version-controlled. Set your token in **Kaggle → Add-ons → Secrets**.

### ⚠️ Save LoRA adapters, not the full model
LoRA adapters are only ~50–200 MB vs ~6 GB for the full merged model. Save the adapters and re-merge with the base model at inference time.

In [1]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
print('✅ Unsloth installed')

## 📦 Cell 1: Imports
Import all required libraries for inference and evaluation. This includes logging suppression utilities, numerical computation libraries (`numpy`, `pandas`), progress tracking (`tqdm`), and PyTorch memory management tools.

In [2]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import os
import re
import json
import logging
import warnings
import gc

import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
from unsloth import FastLanguageModel
import kagglehub

# ── ADD: Dataset path here so all cells can access it ─────────────────────────
DATASET_PATH = "/kaggle/input/datasets/mhariyan/eps-forecast-dataset/final_verified_dataset (1).jsonl"


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## 🤖 Cell 2: Load Fine-Tuned Model
Download the fine-tuned LoRA adapter from Kaggle's model registry using `kagglehub` and load it into memory via Unsloth's `FastLanguageModel`. The model is loaded in **4-bit quantisation** to fit within the T4 GPU's 14.5 GB VRAM limit. Once loaded, it is switched to inference mode which disables dropout and gradient tracking for faster generation.

In [3]:
import kagglehub
from unsloth import FastLanguageModel

MODEL_PATH = kagglehub.model_download("mhariyan/eps_model/keras/default2")
print(f"Model downloaded to: {MODEL_PATH}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_PATH,
    max_seq_length = 2048,   # ← FIXED from 512
    load_in_4bit   = True,
    dtype          = None,
)

FastLanguageModel.for_inference(model)
print("✅ Model loaded and ready for inference!")

Model downloaded to: /kaggle/input/models/mhariyan/eps_model/keras/default2/1
==((====))==  Unsloth 2026.3.8: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.3.8 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✅ Model loaded and ready for inference!


## 📂 Cell 3: Load Dataset
Load the evaluation dataset from the Kaggle input directory. Each record is a JSONL line containing three fields — `instruction` (the task description), `input` (historical financial data), and `output` (the ground truth EPS forecast). All 677 samples are loaded into memory as a list of dictionaries.

In [4]:
# ── CELL 3: Load dataset ──────────────────────────────────────────────────────
samples = []
with open(DATASET_PATH, "r") as f:
    for line in f:
        samples.append(json.loads(line.strip()))

print(f"Loaded {len(samples)} samples")

Loaded 677 samples


## 🛠️ Cell 4: Helper Functions
Define three core utility functions used throughout inference and evaluation:

- **`parse_forecast(text)`** — Strictly extracts the numeric EPS value after the keyword `Forecast:` from model output. Rejects any output where the word `Forecast` is missing, not followed by a valid number, or produces an extreme/unrealistic value.
- **`parse_actual(text)`** — Applies the same parser to the ground truth output field to extract the true EPS value.
- **`get_prev_eps(text)`** — Scans the input text and extracts the most recently known EPS value, used later to compute Directional Accuracy (did the model predict the right UP/DOWN trend?).

In [ ]:
# ── CELL 4: Helper functions ──────────────────────────────────────────────────

ALPACA_PROMPT = """Below is an instruction that describes a task, paired with an input \
that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
"""

def parse_forecast(text: str):
    """
    Clean parser for the retrained model.
    New model stops immediately after 'Forecast: XX.XX' due to EOS training.
    So strict regex on last 5 lines is sufficient — no lenient fallback needed.
    """
    if not text or not isinstance(text, str):
        return None

    # Strip <think> blocks if present
    clean = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL | re.IGNORECASE).strip()
    if not clean:
        clean = text

    # Search last 5 lines — new model always ends with Forecast: XX.XX
    last_lines = "\n".join(clean.strip().splitlines()[-5:])

    pattern = r'\bforecast\b\s*[:\=]\s*(-?\d+(?:\.\d+)?)'
    match   = re.search(pattern, last_lines, re.IGNORECASE)
    if match:
        value = float(match.group(1))
        if 1900 <= value <= 2100:    # reject years
            return None
        if abs(value) > 1_000_000:   # reject extremes
            return None
        return value

    return None

def parse_actual(output_text: str):
    if not output_text or not isinstance(output_text, str):
        return None
    match = re.search(r'\bforecast\b\s*[:\=]\s*(-?\d+(?:\.\d+)?)', output_text, re.IGNORECASE)
    return float(match.group(1)) if match else None

def get_prev_eps(input_text: str):
    matches = re.findall(r'EPS:\s*(-?\d+(?:\.\d+)?)', input_text)
    return float(matches[-1]) if matches else None

## 🚀 Cell 5: Run Batched Inference
Run the fine-tuned model across all valid samples using optimised batched generation. Key optimisations applied:

- **Greedy decoding** (`do_sample=False`) — fastest possible token selection, no stochastic sampling needed for a numeric forecast task
- **Sorted batching** — samples are sorted by input length before batching so similar-length inputs are grouped together, minimising padding waste
- **KV caching** (`use_cache=True`) — reuses key-value attention states across generation steps
- **Reduced output length** (`max_new_tokens=150`) — the model only needs a few tokens to write `Forecast: XX.XX`, so 150 is a safe upper bound
- **OOM fallback** — if a batch causes an out-of-memory error, it automatically retries each sample individually

Each generated output is parsed using `parse_forecast`. Outputs that pass are stored in `results`. Outputs that fail (missing forecast word, no number, empty output) are stored in `rejected` with a labelled reason.

# DEBUG For Cell 5

In [9]:
# ── CELL 5a: Setup only (no full loop yet) ───────────────────────────────────
logging.getLogger("transformers.modeling_attn_mask_utils").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

BATCH_SIZE = 2

def run_inference_batch(batch_samples: list):
    prompts = [
        ALPACA_PROMPT.format(instruction=s["instruction"], input=s["input"])
        for s in batch_samples
    ]
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer(
        prompts,
        return_tensors = "pt",
        padding        = True,
        truncation     = True,
        max_length     = 2048,
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = 800,   # ← was 150, FAR too small for full analysis
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id,
            eos_token_id   = tokenizer.eos_token_id,
            use_cache      = True,
        )

    generated_texts = []
    for i, output in enumerate(outputs):
        new_tokens = output[inputs["input_ids"].shape[1]:]
        generated_texts.append(
            tokenizer.decode(new_tokens, skip_special_tokens=True)
        )
    return generated_texts

# Pre-filter valid samples
results       = []
rejected      = []
skipped_no_gt = 0
valid_samples = []

for i, sample in enumerate(samples):
    actual = parse_actual(sample["output"])
    if actual is None:
        skipped_no_gt += 1
        continue
    sample["_index"]  = i
    sample["_actual"] = actual
    valid_samples.append(sample)

valid_samples.sort(key=lambda s: len(s["input"]))

print(f"Total          : {len(samples)}")
print(f"Skipped (no GT): {skipped_no_gt}")
print(f"Valid to run   : {len(valid_samples)}")
print("✅ Setup complete — ready for debug or full run")

Total          : 677
Skipped (no GT): 8
Valid to run   : 669
✅ Setup complete — ready for debug or full run


In [10]:
# ── CELL 5b: DEBUG — check 3 samples before full run ─────────────────────────
print("Testing first 3 samples...\n")

test_outputs = run_inference_batch(valid_samples[:3])

for i, (sample, generated) in enumerate(zip(valid_samples[:3], test_outputs)):
    print(f"{'='*60}")
    print(f"Sample {i+1}  : {sample.get('source_file', '')}")
    print(f"Actual EPS   : {sample['_actual']}")
    print(f"Raw Output   :\n{generated}")
    print(f"Parsed Value : {parse_forecast(generated)}")
    print()

Testing first 3 samples...

Sample 1  : SBILIFE.NS_RAW.csv
Actual EPS   : 18.9
Raw Output   :
Analysis:
Step 1: Revenue Analysis
To forecast the revenue for 2024, we first need to analyze the Year-over-Year (YoY) growth rate from 2022 to 2023. 
The revenue in 2022 was 829.11 Billion and in 2023 was 806.39 Billion. 
The growth rate can be calculated as: (806.39 - 829.11) / 829.11 = -0.028 or -2.8%.
Given this decline, we might assume a slightly less severe decline for 2024, considering potential market adjustments. 
Let's assume a decline of -1.5% for 2024.
Next Revenue = 806.39 * (1 - 0.015) = 806.39 * 0.985 = 793.00 Billion.

Step 2: Operating Margin
Historically, the operating income has been 0, which implies an operating margin of 0%. 
We will apply this margin to the new revenue to get the forecasted operating income: 
Forecasted Operating Income = 793.00 * 0 = 0.

Step 3: Interest & Tax
Since there's no debt, the interest expense is 0. 
The tax rate can be estimated from the histo

In [ ]:
# ── CELL 5c: Full inference loop ──────────────────────────────────────────────
logging.getLogger("transformers.modeling_attn_mask_utils").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

BATCH_SIZE = 2

def run_inference_batch(batch_samples: list):
    prompts = [
        ALPACA_PROMPT.format(instruction=s["instruction"], input=s["input"])
        for s in batch_samples
    ]
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer(
        prompts,
        return_tensors = "pt",
        padding        = True,
        truncation     = True,
        max_length     = 2048,
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = 800,   # ← new model stops at Forecast: line early
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id,
            eos_token_id   = tokenizer.eos_token_id,
            use_cache      = True,
        )

    generated_texts = []
    for i, output in enumerate(outputs):
        new_tokens = output[inputs["input_ids"].shape[1]:]
        generated_texts.append(
            tokenizer.decode(new_tokens, skip_special_tokens=True)
        )
    return generated_texts

# ── Pre-filter ────────────────────────────────────────────────────────────────
results       = []
rejected      = []
skipped_no_gt = 0
valid_samples = []

for i, sample in enumerate(samples):
    actual = parse_actual(sample["output"])
    if actual is None:
        skipped_no_gt += 1
        continue
    sample["_index"]  = i
    sample["_actual"] = actual
    valid_samples.append(sample)

valid_samples.sort(key=lambda s: len(s["input"]))

print(f"Total          : {len(samples)}")
print(f"Skipped (no GT): {skipped_no_gt}")
print(f"Valid to run   : {len(valid_samples)}")
print(f"\n{'─'*55}")
print(f"  {'#':<6} {'Source File':<30} {'Actual':>10} {'Predicted':>10}")
print(f"{'─'*55}")

counter = 0

for batch_start in range(0, len(valid_samples), BATCH_SIZE):
    batch = valid_samples[batch_start : batch_start + BATCH_SIZE]

    try:
        generated_texts = run_inference_batch(batch)

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"\n⚠️  OOM at batch {batch_start} — dropping to batch size 1...")
            gc.collect()
            torch.cuda.empty_cache()
            generated_texts = []
            for s in batch:
                tokenizer.padding_side = "right"
                inp = tokenizer(
                    [ALPACA_PROMPT.format(instruction=s["instruction"], input=s["input"])],
                    return_tensors="pt", truncation=True, max_length=2048
                ).to("cuda")
                with torch.no_grad():
                    out = model.generate(
                        **inp,
                        max_new_tokens = 800,
                        do_sample      = False,
                        pad_token_id   = tokenizer.eos_token_id,
                        use_cache      = True,
                    )
                generated_texts.append(
                    tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
                )
        else:
            raise e

    for sample, generated in zip(batch, generated_texts):
        counter    += 1
        predicted   = parse_forecast(generated)
        prev_eps    = get_prev_eps(sample["input"])
        source      = sample.get("source_file", "")[:28]
        actual_val  = sample["_actual"]

        if predicted is None:
            reject_reason = "unknown"
            if not generated or generated.strip() == "":
                reject_reason = "Empty output"
            elif not re.search(r'\bforecast\b', generated, re.IGNORECASE):
                reject_reason = "Word 'Forecast' not found"
            else:
                reject_reason = "Forecast found but not followed by a number"

            print(f"  {counter:<6} {source:<30} {actual_val:>10.2f} {'NaN':>10}  ⚠ {reject_reason}")
            rejected.append({
                "index"        : sample["_index"],
                "source_file"  : sample.get("source_file", ""),
                "actual"       : actual_val,
                "raw_output"   : generated[:500],
                "reject_reason": reject_reason,
            })
        else:
            print(f"  {counter:<6} {source:<30} {actual_val:>10.2f} {predicted:>10.2f}")
            results.append({
                "index"      : sample["_index"],
                "source_file": sample.get("source_file", ""),
                "actual"     : actual_val,
                "predicted"  : predicted,
                "prev_eps"   : prev_eps,
                "raw_output" : generated,
            })

# ── Summary ───────────────────────────────────────────────────────────────────
total      = len(samples)
accepted   = len(results)
n_rejected = len(rejected)

print(f"\n{'='*55}")
print(f"  Total                      : {total}")
print(f"  Skipped (no ground truth)  : {skipped_no_gt}")
print(f"  ✅ Accepted                : {accepted}  ({accepted/total*100:.1f}%)")
print(f"  ❌ Rejected                : {n_rejected}  ({n_rejected/total*100:.1f}%)")
print(f"{'='*55}")

if rejected:
    reason_counts = pd.Series([r["reject_reason"] for r in rejected]).value_counts()
    print("\n  Rejection breakdown:")
    for reason, count in reason_counts.items():
        print(f"    • {reason:<45} : {count}")

## 📊 Cell 6: Evaluation Metrics
Compute five evaluation metrics across all accepted predictions and print a full summary report:

| Metric | Description |
|---|---|
| **MSE** (Mean Squared Error) | Penalises large errors heavily — sensitive to outliers |
| **MAE** (Mean Absolute Error) | Average absolute error in raw EPS units |
| **MdAPE** (Median Absolute Percentage Error) | Percentage error robust to outliers — skips zero actuals |
| **MAAPE** (Mean Arctangent Absolute Percentage Error) | Bounded percentage error metric — handles zero actuals safely via arctan |
| **Directional Accuracy** | Percentage of samples where the model correctly predicted whether EPS went UP or DOWN vs the previous year |

Results and per-sample errors are saved to two CSV files:
- `inference_results.csv` — all accepted predictions with error columns
- `rejected_samples.csv` — all bypassed outputs with rejection reasons

In [ ]:
# ── CELL 6: Calculate all evaluation metrics ──────────────────────────────────
import numpy as np
import pandas as pd

if len(results) == 0:
    print("❌ No accepted results to evaluate. Check your rejected samples.")
else:
    df = pd.DataFrame(results)

    y_true = df["actual"].values
    y_pred = df["predicted"].values

    # ── MSE ───────────────────────────────────────────────────────────────────
    mse = np.mean((y_true - y_pred) ** 2)

    # ── MAE ───────────────────────────────────────────────────────────────────
    mae = np.mean(np.abs(y_true - y_pred))

    # ── MdAPE (Median Absolute Percentage Error) ──────────────────────────────
    # Skips samples where actual = 0 to avoid division by zero
    nonzero_mask = y_true != 0
    mdape = np.median(
        np.abs((y_true[nonzero_mask] - y_pred[nonzero_mask]) / y_true[nonzero_mask])
    ) * 100

    # ── MAAPE (Mean Arctangent Absolute Percentage Error) ─────────────────────
    # Bounded between 0–90°, handles zero actuals safely via arctan
    maape = np.mean(
        np.arctan(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-8)))
    ) * 100

    # ── Directional Accuracy ──────────────────────────────────────────────────
    # Did the model predict the right UP/DOWN direction vs previous year EPS?
    da_df      = df[df["prev_eps"].notna() & (df["prev_eps"] != 0)]
    actual_dir = np.sign(da_df["actual"].values   - da_df["prev_eps"].values)
    pred_dir   = np.sign(da_df["predicted"].values - da_df["prev_eps"].values)
    dir_acc    = np.mean(actual_dir == pred_dir) * 100

    # ── Print results ─────────────────────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"  📊 EVALUATION METRICS  ({len(results)} accepted samples)")
    print(f"{'='*55}")
    print(f"  MSE                  : {mse:.4f}")
    print(f"  MAE                  : {mae:.4f}")
    print(f"  MdAPE                : {mdape:.2f}%")
    print(f"  MAAPE                : {maape:.2f}%")
    print(f"  Directional Accuracy : {dir_acc:.2f}%  ({len(da_df)} samples)")
    print(f"{'='*55}")
    print(f"\n  ✅ Accepted  : {accepted}/{total}  ({accepted/total*100:.1f}%)")
    print(f"  ❌ Rejected  : {n_rejected}/{total}  ({n_rejected/total*100:.1f}%)")
    print(f"{'='*55}")

    # ── Save to CSV ───────────────────────────────────────────────────────────
    df["abs_error"]   = np.abs(df["actual"] - df["predicted"])
    df["pct_error"]   = np.where(
        df["actual"] != 0,
        np.abs((df["actual"] - df["predicted"]) / df["actual"]) * 100,
        np.nan
    )
    df["correct_dir"] = (
        np.sign(df["actual"] - df["prev_eps"]) ==
        np.sign(df["predicted"] - df["prev_eps"])
    )

    df.drop(columns=["raw_output"]).to_csv("/kaggle/working/inference_results.csv", index=False)
    pd.DataFrame(rejected).to_csv("/kaggle/working/rejected_samples.csv", index=False)

    print("\n  ✅ Saved: /kaggle/working/inference_results.csv")
    print("  ✅ Saved: /kaggle/working/rejected_samples.csv")